In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
# 1. Loading the cleaned data of Zueriwieneu requests, neighbourhoods and population

reports_gdf = gpd.read_file("../data/cleaned/stzh.zwn_meldungen_p_cleaned.gpkg")
pop_quartiere_gdf = gpd.read_file("../data/cleaned/pop_quartiere_2013_2025.gpkg")

# 2. Assign reports to their nearest neighbourhoods (a few lie just directly outside the city boarder, thus "nearest" was used instead of "within" or "intersects")

reports_quartier = gpd.sjoin_nearest(
    reports_gdf,
    pop_quartiere_gdf,
    how="left"
)

print(f"{reports_quartier['qname'].isna().sum()} reports were not assigned to a neighbourhood")

0 reports were not assigned to a neighbourhood


In [3]:
# (1) How does number of reports evolve over the years, diurnal and across neighbourhoods?

# Extract year
reports_gdf["year"] = (
    reports_gdf["requested_datetime"].dt.year
)

# (1.1) Count the total number of reports per year
report_counts_year = (
    reports_gdf
    .groupby("year")
    .size()
    .reset_index(name="report_count")
)

print("Note: Only 2014-2025 are whole years with reports\n")
display(report_counts_year)

# Extract hour of day
reports_gdf["hour"] = reports_gdf["requested_datetime"].dt.hour

# (1.2) Count the total number of reports per hour
report_counts_hour = (
    reports_gdf
    .groupby("hour")
    .size()
    .reset_index(name="report_count")
)

display(report_counts_hour)

# (1.3) Count total reports per neighbourhood normalized by mean population from 2013-2025

# Asses mean population over the reporting period
population_cols = [
    "2013", "2014", "2015", "2016", "2017",
    "2018", "2019", "2020", "2021", "2022",
    "2023", "2024", "2025"
]

pop_quartiere_gdf["mean_pop_2013-2025"] = (
    pop_quartiere_gdf[population_cols]
    .mean(axis=1)
)
report_counts_quartier = (
    reports_quartier
    .groupby("qname")
    .size()
    .reset_index(name="report_count")
)

# Join counts back to neighbourhood polygons/population
quartier_analysis = pop_quartiere_gdf.merge(
    report_counts_quartier,
    on="qname",
    how="left"
)

# Calculate reports per 1000 inhabitants
quartier_analysis["reports_per_1000"] = (
    quartier_analysis["report_count"]
    / quartier_analysis["mean_pop_2013-2025"]
) * 1000

display(
    quartier_analysis[
        ["qname", "report_count", "mean_pop_2013-2025", "reports_per_1000"]
    ]
    .sort_values("reports_per_1000", ascending=False)
)

Note: Only 2014-2025 are whole years with reports



,year,report_count
0,2013,2901
1,2014,2139
2,2015,1943
3,2016,2307
4,2017,2869
5,2018,3724
6,2019,5244
7,2020,4804
8,2021,5783
9,2022,6828


,hour,report_count
0,0,860
1,1,383
2,2,201
3,3,165
4,4,108
5,5,271
6,6,1085
7,7,2705
8,8,4318
9,9,5026


,qname,report_count,mean_pop_2013-2025,reports_per_1000
15,City,1739,805.461538,2159.010601
30,Hochschulen,1415,670.615385,2110.002294
24,Lindenhof,1124,997.692308,1126.599846
2,Langstrasse,6175,11509.538462,536.511522
13,Rathaus,1454,3266.923077,445.067106
27,Werd,1522,4503.307692,337.973797
4,Enge,2739,9606.384615,285.122875
3,Escher Wyss,1558,5813.076923,268.016409
17,Sihlfeld,5216,21547.692308,242.067685
6,Seefeld,1279,5406.846154,236.551950


In [4]:
# (2) Which neighbourhoods make the most detailed reports (average number of words)?

# Count words in each report detail
reports_quartier["word_count"] = (
    reports_quartier["detail"]
    .str.split() # splits detail in a list of single words
    .str.len() # counts length of the list
)

# Average words per neighbourhood
detail_by_quartier = (
    reports_quartier
    .groupby("qname")["word_count"]
    .mean()
    .reset_index(name="avg_word_count")
    .sort_values("avg_word_count", ascending=False)
)

display(detail_by_quartier)

,qname,avg_word_count
31,Wipkingen,27.295172
24,Schwamendingen-Mitte,23.226384
9,Gewerbeschule,22.011236
6,Escher Wyss,21.329268
32,Witikon,20.708850
15,Höngg,20.626135
21,Oerlikon,19.409274
5,Enge,19.186564
1,Albisrieden,18.923908
28,Unterstrass,18.665366


In [5]:
# (3) How long does the processing of the reports by the council take? Did it change over time and neighbourhoods? (only for reports with "status"="fixed - council")

# Define only the reports that were fixed by the municipal offices
fixed_reports = reports_quartier[
    reports_quartier["status"] == "fixed - council"
].copy()

# Calculate processing duration for these reports
fixed_reports["processing_time"] = (
    fixed_reports["updated_datetime"]
    - fixed_reports["requested_datetime"]
)

# recalcualte processing duration to decimal days
fixed_reports["processing_days"] = (
    fixed_reports["processing_time"]
    .dt.total_seconds() / (60 * 60 * 24)
)

# Extract year
fixed_reports["year"] = (
    fixed_reports["requested_datetime"].dt.year
)

# (3.1) Count the average processing duration per year
processing_time_year = (
    fixed_reports
    .groupby("year")["processing_days"]
    .mean()
    .reset_index(name="avg_processing_days")
    .sort_values("year", ascending=True)
)

display(processing_time_year)

# (3.2) Count the average processing duration per neighbourhood
processing_time_by_quartier = (
    fixed_reports
    .groupby("qname")["processing_days"]
    .mean()
    .reset_index(name="avg_processing_days")
    .sort_values("avg_processing_days", ascending=True)
)

display(processing_time_by_quartier)

,year,avg_processing_days
0,2013,3.061863
1,2014,2.444667
2,2015,3.049618
3,2016,3.000544
4,2017,2.540313
5,2018,2.707137
6,2019,1.832676
7,2020,2.906078
8,2021,3.155192
9,2022,3.265625


,qname,avg_processing_days
30,Werd,1.923301
12,Hirzenbach,2.032798
16,Langstrasse,2.122421
27,Sihlfeld,2.153757
10,Hard,2.205390
26,Seefeld,2.769765
11,Hirslanden,2.782081
19,Mühlebach,2.829791
0,Affoltern,2.890134
15,Höngg,3.001785


In [7]:
# (4) What percentage of reports in a neighbourhood belong to each issue type?
# Count issue types per neighbourhood
issue_by_quartier = (
    reports_quartier
    .groupby(["qname", "service_name"])
    .size()
    .reset_index(name="report_count")
)

# Calculate absolute number of total reports per neighbourhood
quartier_totals = (
    reports_quartier
    .groupby("qname")
    .size()
    .reset_index(name="total_reports")
)

# Join totals back
issue_by_quartier = issue_by_quartier.merge(
    quartier_totals,
    on="qname",
    how="left"
)

# Calculate share of each issue category for each neighbourhood
issue_by_quartier["share_of_reports"] = (
    issue_by_quartier["report_count"]
    / issue_by_quartier["total_reports"]
) * 100

issue_pivot = issue_by_quartier.pivot(
    index="qname",
    columns="service_name",
    values="share_of_reports"
).fillna(0)

display(issue_pivot)

service_name,Abfall/Sammelstelle,Allgemein,Beleuchtung/Uhren,Brunnen/Hydranten,Graffiti,Grünflächen/Spielplätze,Schädlinge,Signalisation/Lichtsignal,Strasse/Trottoir/Platz,VBZ/ÖV
qname,,,,,,,,,,
Affoltern,32.197757,4.528459,10.178646,1.246365,5.525550,12.796012,1.661820,15.330287,13.585376,2.949730
Albisrieden,31.516937,5.252823,6.676485,3.240059,2.601865,12.911144,2.209131,16.789396,16.445754,2.356406
Alt-Wiedikon,51.062124,5.891784,6.733467,1.122244,1.362725,5.851703,1.082164,13.226453,11.823647,1.843687
Altstetten,34.167280,6.181015,5.862154,3.213147,4.341428,8.118715,1.937699,19.009075,14.471425,2.698062
City,25.071880,7.993099,6.267970,1.782634,5.520414,5.462910,0.230017,21.506613,21.909143,4.255319
Enge,25.155166,9.784593,7.265425,1.496897,8.470245,9.200438,0.693684,19.897773,14.968967,3.066813
Escher Wyss,23.042362,3.337612,6.161746,1.155327,5.648267,9.050064,0.513479,21.822850,25.224647,4.043646
Fluntern,31.254996,5.755396,9.512390,3.597122,5.355715,12.789768,1.678657,14.228617,13.669065,2.158273
Friesenberg,23.650568,5.326705,8.948864,2.130682,4.048295,28.551136,2.627841,14.062500,9.801136,0.852273


In [ ]:
# (5) At which neighbourhoods do people care about road markings and traffic signs (use LLM to assess which wordings are searched in descriptions and titles)?